# 00 - Pipeline Lengkap (end-to-end)

Notebook ringkas yang menjalankan seluruh tahap secara berurutan. Untuk penjelasan
tiap tahap, gunakan notebook 01-04.

## 1. Setup

In [1]:
import os, sys, json
from pathlib import Path
from IPython.display import display, Image

# Temukan root proyek (folder berisi configs/experiment.yaml), lalu jadikan cwd.
_cands = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
ROOT = next(p for p in _cands if (p / "configs" / "experiment.yaml").exists())
os.chdir(ROOT)
for _p in (ROOT / "src", ROOT / "scripts"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from vehicle_lighting.config import load_config, resolve
cfg = load_config()
print("ROOT   :", ROOT)
print("config :", ROOT / "configs" / "experiment.yaml")
print("kelas  :", cfg["data"]["classes"])
print("model  :", cfg["models"])

ROOT   : D:\pkac\kendaraan-pencahayaan-mobilenet-resnet
config : D:\pkac\kendaraan-pencahayaan-mobilenet-resnet\configs\experiment.yaml
kelas  : ['mobil', 'sepeda_motor', 'bus', 'truk']
model  : ['mobilenet_v2', 'resnet50']


## 2. Bangun dataset (lewati bila `data/manifest.csv` sudah ada)

In [3]:
manifest = resolve(cfg, cfg["data"]["manifest"])
if not manifest.exists():
    import subprocess
    cmd = [sys.executable, "scripts/bdd100k_to_dataset.py",
           "--bdd-root", str(cfg["data"]["bdd100k"]["root"]),
           "--per-class", str(cfg["data"]["bdd100k"]["per_class"]),
           "--seed", str(cfg["project"]["seeds"][0])]
    print(" ".join(cmd))
    r = subprocess.run(cmd, text=True, capture_output=True)
    print(r.stdout[-3000:])
else:
    print("manifest sudah ada:", manifest)

C:\Users\ASUS\miniconda3\envs\kendaraan-pencahayaan\python.exe scripts/bdd100k_to_dataset.py --bdd-root ../bdd100k/bdd100k --per-class 3000 --seed 42


KeyboardInterrupt: 

## 3. Verifikasi pencahayaan

In [ ]:
import cv2, pandas as pd
from vehicle_lighting.lighting import mean_luminance
df = pd.read_csv(resolve(cfg, cfg["data"]["manifest"]))
df["luminance"] = [mean_luminance(cv2.cvtColor(cv2.imread(str(resolve(cfg, p))),
                                                cv2.COLOR_BGR2RGB)) for p in df["path"]]
print(df.groupby("lighting")["luminance"].mean())

## 4. Latih + evaluasi (smoke test dulu, lalu matriks penuh)

In [ ]:
from vehicle_lighting.train import run
run(cfg, "mobilenet_v2", "normal", cfg["project"]["seeds"][0], epochs=1, pretrained=False)

In [ ]:
results_dir = resolve(cfg, cfg["results_dir"])
for reg in ["normal", "mixed"]:
    for m in cfg["models"]:
        for s in cfg["project"]["seeds"]:
            if (results_dir / f"{m}__{reg}__seed{s}" / "metrics.json").exists():
                print("lewati:", m, reg, s); continue
            run(cfg, m, reg, s, None, None, save_ckpt=True)

## 5. Agregasi, visualisasi, laporan

In [ ]:
from vehicle_lighting import aggregate, report
aggregate.main()
report.main()
print((resolve(cfg, "docs/laporan/laporan.md")).read_text(encoding="utf-8")[:1200])